# Quick Start: Train a 3D U-Net on Synthetic Data

This notebook walks through the core CosmoRecon workflow:

1. Generate synthetic 3D density fields and a binary mask
2. Build a `tf.data.Dataset` pipeline with `create_dataset`
3. Construct a U-Net model
4. Train for a few epochs with periodic checkpointing
5. Run inference and inspect the output

Everything runs on CPU with small 16³ volumes so there is no GPU requirement.

In [ ]:
import os, tempfile, glob
import numpy as np
import tensorflow as tf

from CosmoRecon import (
    UNet,
    create_dataset,
    SaveEveryNEpoch,
)

## 1. Generate synthetic data

We create 10 random 16³ "observed" and "true" `.npy` files in a temporary
directory, plus a single binary mask that zeroes out a cube in the centre
of the volume.

In [ ]:
FIELD_SIZE = 16
N_FILES = 10
NORM_VAL = 1.0          # synthetic data is already in [0, 1]
BASE_FILTERS = 4        # keep the model tiny for CPU
MIN_SIZE = 4

tmp = tempfile.mkdtemp(prefix="cosmorecon_quickstart_")
obs_dir  = os.path.join(tmp, "observed")
true_dir = os.path.join(tmp, "true")
mask_dir = os.path.join(tmp, "masks")
out_dir  = os.path.join(tmp, "output")

for d in [obs_dir, true_dir, mask_dir, out_dir]:
    os.makedirs(d)

rng = np.random.default_rng(42)
for i in range(N_FILES):
    obs  = rng.random((FIELD_SIZE,) * 3, dtype=np.float32)
    true = rng.random((FIELD_SIZE,) * 3, dtype=np.float32)
    np.save(os.path.join(obs_dir,  f"field_{i:03d}.npy"), obs)
    np.save(os.path.join(true_dir, f"field_{i:03d}.npy"), true)

mask = np.ones((FIELD_SIZE,) * 3, dtype=np.float32)
mask[4:12, 4:12, 4:12] = 0.0           # central cubic hole
np.save(os.path.join(mask_dir, "mask.npy"), mask)

print(f"Data written to {tmp}")
print(f"  Observed: {len(os.listdir(obs_dir))} files")
print(f"  True:     {len(os.listdir(true_dir))} files")
print(f"  Mask:     {mask.shape}, fraction missing = {1 - mask.mean():.2%}")

## 2. Build a `tf.data.Dataset`

In [ ]:
x_files = sorted(glob.glob(os.path.join(obs_dir,  "*.npy")))
y_files = sorted(glob.glob(os.path.join(true_dir, "*.npy")))

train_ds = create_dataset(
    x_files, y_files,
    batch_size=4,
    field_size=FIELD_SIZE,
    norm_val=NORM_VAL,
    shuffle=True,
)

# Inspect one batch
for xb, yb in train_ds.take(1):
    print(f"x batch shape: {xb.shape}  (batch, D, H, W, channels)")
    print(f"y batch shape: {yb.shape}")
    print(f"x range: [{xb.numpy().min():.4f}, {xb.numpy().max():.4f}]")

## 3. Build and compile the model

In [ ]:
builder = UNet(
    base_filters=BASE_FILTERS,
    min_size=MIN_SIZE,
    input_field="rho",
    norm_val=NORM_VAL,
)

model = builder.prepare_model(
    input_size=(FIELD_SIZE, FIELD_SIZE, FIELD_SIZE, 1)
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="mse",
)

print(f"Parameters: {model.count_params():,}")
model.summary()

## 4. Train with periodic checkpointing

`SaveEveryNEpoch` saves the model every *period* epochs.

In [ ]:
ckpt_dir = os.path.join(out_dir, "checkpoints")
os.makedirs(ckpt_dir, exist_ok=True)

checkpoint_cb = SaveEveryNEpoch(
    filepath=os.path.join(ckpt_dir, "model_{epoch:03d}.keras"),
    period=2,
)

history = model.fit(
    train_ds,
    epochs=4,
    callbacks=[checkpoint_cb],
    verbose=1,
)

saved = sorted(os.listdir(ckpt_dir))
print(f"\nSaved checkpoints: {saved}")

## 5. Inference on a single sample

In [ ]:
sample = np.load(x_files[0]).astype(np.float32) / NORM_VAL
sample = sample[np.newaxis, ..., np.newaxis]   # (1, D, H, W, 1)

pred = model.predict(sample, verbose=0)
print(f"Input  shape: {sample.shape}  range: [{sample.min():.4f}, {sample.max():.4f}]")
print(f"Output shape: {pred.shape}  range: [{pred.min():.4f}, {pred.max():.4f}]")

## 6. Visualise a central slice

In [ ]:
import matplotlib.pyplot as plt

mid = FIELD_SIZE // 2
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))

titles = ["Input", "Target", "Prediction"]
target = np.load(y_files[0]).astype(np.float32) / NORM_VAL
slices = [
    sample[0, :, :, mid, 0],
    target[:, :, mid],
    pred[0, :, :, mid, 0],
]

for ax, title, s in zip(axes, titles, slices):
    im = ax.imshow(s, origin="lower", cmap="inferno")
    ax.set_title(title)
    plt.colorbar(im, ax=ax, fraction=0.046)

fig.suptitle(f"Central z-slice (z = {mid})")
plt.tight_layout()
plt.show()